In [1]:
from forget.api import InstructorLLM
from pydantic import BaseModel, field_validator
import pandas as pd
from pathlib import Path
import random
from sklearn.model_selection import train_test_split

In [2]:
PRESIDENTS = ["Barack Obama", "Donald Trump", "Joe Biden"]
RAW_CSV = Path("store/presidents/presidents_raw.csv")
OUT_CSV = Path("store/presidents/presidents.csv")

categories = open("store/presidents/mini.csv").read().strip().splitlines()

# ---- test controls ----
MAX_PRESIDENTS = None   # set to None for all
MAX_CATS = None        # set to None for all

presidents = PRESIDENTS[:MAX_PRESIDENTS]
cats = categories[:MAX_CATS]

## Generation

In [2]:
llm = InstructorLLM(provider_model="openai/gpt-5-nano", concurrency=500)

In [3]:
class Question(BaseModel):
    q: str
    a: str
    b: str
    c: str
    d: str
    ans: str

    @field_validator("ans")
    @classmethod
    def ans_must_be_valid(cls, v):
        if v not in ("a", "b", "c", "d"):
            raise ValueError(f"ans must be a/b/c/d, got {v}")
        return v


class QuestionSet(BaseModel):
    q1: Question
    q2: Question
    q3: Question
    q4: Question
    q5: Question
    q6: Question
    q7: Question
    q8: Question
    q9: Question
    q10: Question
    q11: Question
    q12: Question
    q13: Question
    q14: Question
    q15: Question
    q16: Question
    q17: Question
    q18: Question
    q19: Question
    q20: Question

In [ ]:
def make_prompt(president: str, category: str) -> str:
    return (
        f"Generate 20 unique multiple-choice trivia questions about {president} "
        f"specifically related to: {category}.\n\n"
        "Each question must have exactly 4 options (a, b, c, d) and one correct answer. "
        "Make questions specific, factual, and varied in difficulty."
    )


def flatten_to_rows(president: str, category: str, qs: QuestionSet) -> list[dict]:
    return [
        {
            "president": president,
            "cat": category,
            "q": q.q, "a": q.a, "b": q.b, "c": q.c, "d": q.d, "ans": q.ans,
        }
        for q in [getattr(qs, f"q{i}") for i in range(1, 21)]
    ]


def save_csv(all_rows: list[dict]):
    RAW_CSV.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(all_rows).to_csv(RAW_CSV, index=False)

In [7]:
all_rows = []

for president in presidents:
    prompts = [make_prompt(president, cat) for cat in cats]
    models = [QuestionSet] * len(cats)

    results = await llm.batch_respond(
        prompts=prompts,
        response_models=models,
        system="You are a trivia question writer. Return exactly 20 questions.",
        desc=president,
        max_retries=20,
    )

    for cat, result in zip(cats, results):
        all_rows.extend(flatten_to_rows(president, cat, result))

save_csv(all_rows)

Joe Biden: 100%|██████████| 250/250 [17:02<00:00,  4.09s/it] 


In [ ]:
df = pd.read_csv(RAW_CSV)
print(df.shape)
print(df.groupby(["president", "cat"]).size())

(15000, 8)
president     cat                                       
Barack Obama  Abortion policy positions                     20
              Afghanistan policy                            20
              Africa policy                                 20
              Air Force One details                         20
              Approval ratings over time                    20
                                                            ..
Joe Biden     Welfare and safety net policy                 20
              White House entertaining and state dinners    20
              White House staff turnover                    20
              Who administered the oath                     20
              Writing career before presidency              20
Length: 750, dtype: int64


## Fixing

In [14]:
df = pd.read_csv(RAW_CSV)

In [15]:
# Rebalance ans labels to 25% each by swapping option text into a new target slot
slots = ["a", "b", "c", "d"]

indices = list(df.index)
random.shuffle(indices)
target_map = {idx: slots[i % 4] for i, idx in enumerate(indices)}

for idx, row in df.iterrows():
    current = row["ans"]
    target = target_map[idx]
    if current == target:
        continue
    df.at[idx, current], df.at[idx, target] = row[target], row[current]
    df.at[idx, "ans"] = target

In [16]:
df.to_csv(OUT_CSV, index=False)

In [17]:
df["ans"].value_counts(normalize=True).sort_index()

ans
a    0.25
b    0.25
c    0.25
d    0.25
Name: proportion, dtype: float64

## Subsets

In [14]:
# 1. Drop NaN rows
# 2. Group by president, find the smallest group size
# 3. Subsample each president to that size
# 4. GOOD = balanced df with correct associations
#    BAD  = copy with randomly corrupted ans labels
# 5. Save good.csv / bad.csv
# 6. 80:20 train/val split on both, save good_train.csv etc.

STORE = OUT_CSV.parent

In [15]:
# 1-3: drop nans, balance by president
df = pd.read_csv(OUT_CSV).dropna()
min_n = df.groupby("president").size().min()
good = pd.concat(g.sample(min_n) for _, g in df.groupby("president")).reset_index(drop=True)

# 4: corrupt ans for bad set
bad = good.copy()
slots = ["a", "b", "c", "d"]
for idx, row in bad.iterrows():
    wrong = [s for s in slots if s != row["ans"]]
    bad.at[idx, "ans"] = random.choice(wrong)

# 5: save
good.to_csv(STORE / "good.csv", index=False)
bad.to_csv(STORE / "bad.csv", index=False)

In [16]:
# 6: 80:20 train/val split
good_train, good_val = train_test_split(good, test_size=0.2, random_state=42)
bad_train, bad_val = train_test_split(bad, test_size=0.2, random_state=42)

good_train.to_csv(STORE / "good_train.csv", index=False)
good_val.to_csv(STORE / "good_val.csv", index=False)
bad_train.to_csv(STORE / "bad_train.csv", index=False)
bad_val.to_csv(STORE / "bad_val.csv", index=False)

In [17]:
print(min_n)

4938


In [18]:
good

,president,cat,q,a,b,c,d,ans
0,Barack Obama,Use of executive privilege,"During the 2012 Fast and Furious dispute, whic...",Democrats,Republicans,Independents,It alternated,b
1,Barack Obama,Third party candidates in the race,Did any third-party candidate win electoral vo...,Yes,Only one state,No,Several,c
2,Barack Obama,Family separation and detention policy,Which agency operated the Obama-era family det...,FBI,CBP,USCIS,ICE,d
3,Barack Obama,Controversial public statements,Which comment did Barack Obama make about Tray...,We should not discuss the matter in public,Trayvon Martin was innocent and nothing else,Trayvon Martin could have been my son,All criminal cases deserve equal outcomes,c
4,Barack Obama,Federal Reserve relations and appointments,"Jerome Powell, nominated to the Federal Reserv...",Chair of the Council of Economic Advisers,Chair of the Federal Reserve,Deputy Secretary of the Treasury,Secretary of the Treasury,b
...,...,...,...,...,...,...,...,...
14809,Joe Biden,Federal Reserve relations and appointments,True or False: The President alone can appoint...,False,True,Only during wartime,Only if approved by the Supreme Court,a
14810,Joe Biden,Africa policy,Is democracy and governance a stated priority ...,True,False,Not mentioned,Only in elections,a
14811,Joe Biden,Social Security positions,Biden has proposed lifting the payroll tax cap...,Keep the cap at its current level,Lift the cap (apply Social Security tax to all...,Eliminate Social Security taxes entirely,Remove all payroll taxes,b
14812,Joe Biden,Africa policy,What role does Biden's administration propose ...,Promote democracy and governance and anti-corr...,Ignore governance,Impose sanctions without cause,Only seek military alliances,a
